In [10]:
pip install beautifulsoup4 requests


   -------------------- ------------------- 1/2 [beautifulsoup4]
   -------------------- ------------------- 1/2 [beautifulsoup4]
   ---------------------------------------- 2/2 [beautifulsoup4]

Note: you may need to restart the kernel to use updated packages.


In [11]:
# ========= SCRAPING PC COMPONENTES: MONITORES GAMING (PRIMERA PÁGINA) =========

import requests
from bs4 import BeautifulSoup
import pandas as pd

# 1. URL de ejemplo: monitores gaming (ajusta si hace falta)
BASE_URL = "https://www.pccomponentes.com/monitores-gaming"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

def get_page_html(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text

def parse_product_list(html):
    soup = BeautifulSoup(html, "html.parser")
    
    # OJO: estos selectores hay que ajustarlos mirando el HTML real de PCComponentes
    # Abre la web, inspecciona una tarjeta de producto y cambia estas clases si no coinciden.
    product_cards = soup.select("article.sc-product-card")
    print("Nº de tarjetas encontradas dentro de la función:", len(product_cards))
    
    data = []
    for card in product_cards:
        # Nombre del producto
        name_el = card.select_one(".sc-product-card__title")
        name = name_el.get_text(strip=True) if name_el else None
        
        # Precio actual
        price_el = card.select_one(".sc-product-card__price")
        price_text = price_el.get_text(strip=True) if price_el else None
        
        # Precio original (si hay descuento)
        original_el = card.select_one(".sc-product-card__price--previous")
        original_price_text = original_el.get_text(strip=True) if original_el else None
        
        # Rating (estrellas)
        rating_el = card.select_one(".sc-product-card__rating")
        rating = rating_el.get("data-rating") if rating_el else None
        
        # Número de opiniones
        reviews_el = card.select_one(".sc-product-card__reviews")
        reviews_text = reviews_el.get_text(strip=True) if reviews_el else None
        
        data.append({
            "nombre": name,
            "precio_actual_raw": price_text,
            "precio_original_raw": original_price_text,
            "rating_raw": rating,
            "opiniones_raw": reviews_text,
            "url_categoria": BASE_URL
        })
    
    return data

# 2. Ejecutar scraping de la primera página y mostrar resultados
html = get_page_html(BASE_URL)
productos = parse_product_list(html)

print("Nº de productos encontrados:", len(productos))

df = pd.DataFrame(productos)
print(df.head())

Nº de tarjetas encontradas dentro de la función: 0
Nº de productos encontrados: 0
Empty DataFrame
Columns: []
Index: []


In [13]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

BASE_URL = "https://www.pccomponentes.com/monitores-pc"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/123.0.0.0 Safari/537.36"
    )
}

def get_page_html(url):
    response = requests.get(url, headers=headers)
    response.raise_for_status()
    return response.text

def parse_product_list(html):
    soup = BeautifulSoup(html, "html.parser")
    
    product_cards = soup.select("div.product-card")
    print("Nº de tarjetas encontradas dentro de la función:", len(product_cards))
    
    data = []
    for card in product_cards:
        # Nombre
        name_el = card.select_one("a.product-card__name") or card.select_one("a[title]")
        name = name_el.get_text(strip=True) if name_el else None
        
        # Precio actual: buscamos cualquier elemento con "€" dentro del card
        price_el = None
        for el in card.select("*"):
            text = el.get_text(strip=True)
            if "€" in text:
                price_el = el
                break
        price_text = price_el.get_text(strip=True) if price_el else None
        
        # Precio original (si hay precio tachado, suele aparecer distinto)
        original_price_text = None
        # Si ves en el HTML una clase concreta para el precio tachado, cámbiala aquí:
        original_el = card.select_one(".price-old, .product-card__price-regular")
        if original_el:
            original_price_text = original_el.get_text(strip=True)
        
        # Rating y opiniones: de momento dejamos placeholders
        rating_el = card.select_one(".product-card__rating")
        rating_text = rating_el.get_text(strip=True) if rating_el else None
        
        reviews_el = card.select_one(".product-card__reviews")
        reviews_text = reviews_el.get_text(strip=True) if reviews_el else None
        
        data.append({
            "nombre": name,
            "precio_actual_raw": price_text,
            "precio_original_raw": original_price_text,
            "rating_raw": rating_text,
            "opiniones_raw": reviews_text,
            "url_categoria": BASE_URL
        })
    
    return data

# Ejecutar scraping de la primera página y mostrar resultados
html = get_page_html(BASE_URL)
productos = parse_product_list(html)

print("Nº de productos encontrados:", len(productos))

df = pd.DataFrame(productos)
print(df.head())

Nº de tarjetas encontradas dentro de la función: 40
Nº de productos encontrados: 40
  nombre precio_actual_raw precio_original_raw rating_raw opiniones_raw  \
0   None              None                None       None          None   
1   None              None                None       None          None   
2   None              None                None       None          None   
3   None              None                None       None          None   
4   None              None                None       None          None   

                                url_categoria  
0  https://www.pccomponentes.com/monitores-pc  
1  https://www.pccomponentes.com/monitores-pc  
2  https://www.pccomponentes.com/monitores-pc  
3  https://www.pccomponentes.com/monitores-pc  
4  https://www.pccomponentes.com/monitores-pc  
